# 4. Ensembles de Arboles de Decision

## 4.3 Random Forest

*Random Forest* es un algoritmo de ensembles de arboles de decision creado por Leo Brieman en 1995-2006
https://link.springer.com/content/pdf/10.1023/a:1010933404324.pdf

La página original es:
https://www.stat.berkeley.edu/~breiman/RandomForests/cc_home.htm

Dos buenos videos para seguir el paso a paso de Random Forest y aplicaciones:
* https://www.youtube.com/watch?v=J4Wdy0Wc_xQ
* https://www.youtube.com/watch?v=sQ870aTKqiM

Qué tipo de perturbaciones se realizan en Random Forest

*   Se perturba el dataset, con la técnica de bagging = Bootstrap Aggregating
*   Tambien se perturba el algoritmo, utiliza random en cada split

Cada arbolito de Random Forest se entrena sobre un dataset perturbado, que tiene :
* todas las columnas originales (esta es una GRAN diferencia con  Arboles Azarosos)
* la misma *cantidad* de registros del dataset original, PERO generados por la técnica de sampleo con reposición del dataset original.

A pesar de que Leo Brieman es también el inventor de CART (Classification and Regression Trees) Random Forest no corre el algoritmo CART de la libreria rpart, sino un CART perturbado, en donde cada split NO se hace sobre todos los campos del dataset, sino sobre un csubconjunto tomado al azar, esa cantidad es el hiperparámetro *mtry*

#### 4.3.1 Seteo del ambiente local


Esta parte se debe correr con un kernel de R local.
<br>En Jupyter, seleccionar el kernel **R** antes de ejecutar el notebook.


Los archivos persistentes quedan en el repo local: datasets en `datasets/` y resultados en `exp/`.


In [1]:
# Seteo local: resuelve el root del repo (funciona desde raiz, src/ensembles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")


Para correr localmente, el dataset debe estar en `datasets/` dentro del repo.

<br>Si se va a subir a Kaggle, copiar `kaggle.json` a la raiz del repo antes de correr la siguiente celda. La celda lo instala en `~/.kaggle/kaggle.json` con permisos correctos.


In [2]:
dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)

dataset_local <- file.path(DATA_DIR, "dataset_pequeno.csv")
if (!file.exists(dataset_local)) {
  stop("No encuentro el dataset en: ", dataset_local)
}

if (file.exists(KAGGLE_JSON)) {
  kaggle_dir <- path.expand("~/.kaggle")
  dir.create(kaggle_dir, recursive = TRUE, showWarnings = FALSE)
  file.copy(KAGGLE_JSON, file.path(kaggle_dir, "kaggle.json"), overwrite = TRUE)
  Sys.chmod(file.path(kaggle_dir, "kaggle.json"), mode = "0600")
} else {
  message("No encontre kaggle.json en la raiz del repo. Solo es necesario para hacer submit a Kaggle.")
}




---



### 4.4  Random Forest, una corrida

El tiempo de corrida de este punto es de alrededor de 8 minutos

Esta parte se debe correr con el kernel de **R**.


limpio el ambiente de R

In [3]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 08 12:36:35 2026"

In [4]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,676829,36.2,1489772,79.6,NA,1489772,79.6
Vcells,1259246,9.7,8388608,64.0,49152,2014490,15.4


**ranger** es una de las muchas librerías en lenguage R que implementa el algoritmo *Random Forest*, tiene la ventaja que corre el paralelo, utilizando todos los nucleos del procesador.

In [5]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

# ranger se usa para procesar
if( !require("ranger") ) install.packages("ranger")
require("ranger")

# randomForest  solo se usa para imputar nulos
if( !require("randomForest") ) install.packages("randomForest")
require("randomForest")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: ranger

Loading required package: randomForest

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘randomForest’


The following object is masked from ‘package:ranger’:

    importance




Aqui debe cargar SU semilla primigenia y

In [6]:
PARAM <- list()
PARAM$experimento <- 440
PARAM$semilla_primigenia <- 300089

PARAM$semilla_primigenia <- 100003

# PARAM$ranger$num.trees <- 300 # cantidad de arboles
# PARAM$ranger$mtry <- 13 # cantidad de atributos que participan en cada split
# PARAM$ranger$min.node.size <- 50 # tamaño minimo de las hojas
# PARAM$ranger$max.depth <- 10 # 0 significa profundidad infinita


# PARAM$ranger$num.trees     <- 600
# PARAM$ranger$mtry          <- 22      # Higher mtry to compensate for shallow leaves
# PARAM$ranger$min.node.size <- 100     # Aggressive noise suppression
# PARAM$ranger$max.depth     <- 18      # Caps extreme depth while allowing rich splits
# PARAM$ranger$replace       <- FALSE   # Subsampling without replacement
# PARAM$ranger$sample.fraction <- 0.632

# PARAM$ranger$num.trees     <- 500
# PARAM$ranger$mtry          <- 22      # Higher mtry to compensate for shallow leaves
# PARAM$ranger$min.node.size <- 100     # Aggressive noise suppression
# PARAM$ranger$max.depth     <- 18      # Caps extreme depth while allowing rich splits
# PARAM$ranger$replace       <- FALSE   # Subsampling without replacement
# PARAM$ranger$sample.fraction <- 0.632

# PARAM$ranger$num.trees     <- 600
# PARAM$ranger$mtry          <- 15
# PARAM$ranger$min.node.size <- 20
# PARAM$ranger$max.depth     <- 0
# PARAM$ranger$splitrule     <- "hellinger" # Optimized for imbalanced classification

# luciana experimento.
PARAM$ranger$num.trees <- 500
PARAM$ranger$mtry <- 153 # cantidad de atributos que participan en cada split
PARAM$ranger$min.node.size <- 20 # tamaño minimo de las hojas
PARAM$ranger$max.depth <- 18 # 0 significa profundidad infinita



In [7]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento_folder <- paste0("KA", PARAM$experimento)
dir.create(file.path(EXP_DIR, experimento_folder), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento_folder))


In [8]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [9]:
#  estas dos lineas estan relacionadas con el Data Drifting
# asigno un valor muy negativo

if( "Master_Finiciomora" %in% colnames(dataset) )
  dataset[ is.na(Master_Finiciomora) , Master_Finiciomora := -999 ]

if( "Visa_Finiciomora" %in% colnames(dataset) )
  dataset[ is.na(Visa_Finiciomora) , Visa_Finiciomora :=  -999 ]


In [10]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

In [11]:
set.seed( PARAM$semilla_primigenia ) # Establezco la semilla aleatoria


# ranger necesita la clase de tipo factor
factorizado <- as.factor(dtrain$clase_ternaria)
dtrain[, clase_ternaria := factorizado]

In [12]:
# Ranger NO acepta valores nulos
# Leo Breiman, ¿por que le temias a los nulos?
# imputo los nulos, ya que ranger no acepta nulos
dtrain <- na.roughfix(dtrain)

In [13]:
setorder(dtrain, clase_ternaria) # primero quedan los BAJA+1, BAJA+2, CONTINUA

# genero el modelo de Random Forest llamando a ranger()
modelo <- ranger(
  formula= "clase_ternaria ~ .",
  data= dtrain,
  probability= TRUE, # para que devuelva las probabilidades
  num.trees= PARAM$ranger$num.trees,
  mtry= PARAM$ranger$mtry,
  min.node.size= PARAM$ranger$min.node.size,
  max.depth= PARAM$ranger$max.depth
)


Growing trees.. Progress: 4%. Estimated remaining time: 14 minutes, 16 seconds.
Growing trees.. Progress: 7%. Estimated remaining time: 13 minutes, 57 seconds.
Growing trees.. Progress: 11%. Estimated remaining time: 13 minutes, 12 seconds.
Growing trees.. Progress: 15%. Estimated remaining time: 12 minutes, 34 seconds.
Growing trees.. Progress: 18%. Estimated remaining time: 12 minutes, 2 seconds.
Growing trees.. Progress: 22%. Estimated remaining time: 11 minutes, 26 seconds.
Growing trees.. Progress: 26%. Estimated remaining time: 10 minutes, 55 seconds.
Growing trees.. Progress: 29%. Estimated remaining time: 10 minutes, 21 seconds.
Growing trees.. Progress: 33%. Estimated remaining time: 9 minutes, 44 seconds.
Growing trees.. Progress: 37%. Estimated remaining time: 9 minutes, 7 seconds.
Growing trees.. Progress: 41%. Estimated remaining time: 8 minutes, 29 seconds.
Growing trees.. Progress: 45%. Estimated remaining time: 7 minutes, 59 seconds.
Growing trees.. Progress: 49%. Estim

In [14]:
# Carpinteria necesaria sobre  dfuture
# como quiere la Estadistica Clasica, imputar nulos por separado
# ( aunque en este caso ya tengo los datos del futuro de antemano
#  pero bueno, sigamos el librito de estos fundamentalistas a rajatabla ...

dfuture[, clase_ternaria := NULL]
dfuture <- na.roughfix(dfuture)

In [15]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]

In [16]:
# aplico el modelo a los datos que no tienen clase
# aplico el modelo recien creado a los datos del futuro
prediccion <- predict(modelo, dfuture)

tb_prediccion[, prob := prediccion$predictions[, "BAJA+2"] ]

In [17]:
tb_prediccion[, Predicted := as.numeric(prob > (1/40))]

In [18]:
archivo_kaggle <- paste0("LUCIANA", PARAM$experimento,".csv")

# grabo el archivo
fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
 file= archivo_kaggle,
 sep= ","
)


In [ ]:
# # subida a Kaggle
# comando <- "kaggle competitions submit"
# competencia <- "-c data-mining-inicial-2026-b"
# arch <- paste( "-f", archivo_kaggle)

# mensaje <- paste0("-m 'num.trees=", PARAM$ranger$num.trees, "  mtry=", PARAM$ranger$mtry, "  min.node.size=", PARAM$ranger$min.node.size, " max.depth=", PARAM$ranger$max.depth, "'" )
# linea <- paste( comando, competencia, arch, mensaje)
# salida <- system(linea, intern= TRUE)
# cat(salida)

58 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B

In [40]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 08 12:35:13 2026"



---

